# Processing Airline Dataset

In [1]:
import pandas as pd

file_path = '/content/Airline Dataset Updated - v2.csv'
df = pd.read_csv(file_path)
display(df.head())

,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Continents,Departure Date,Arrival Airport,Pilot Name,Flight Status
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,6/28/2022,CXF,Fransisco Hazeldine,On Time
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,12/26/2022,YCO,Marla Parsonage,On Time
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,Europe,1/18/2022,GNB,Rhonda Amber,On Time
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,9/16/2022,YND,Kacie Commucci,Delayed
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2/25/2022,SEE,Ebonee Tree,On Time


In [2]:
df.isnull().sum() #check null value

,0
Passenger ID,0
First Name,0
Last Name,0
Gender,0
Age,0
Nationality,0
Airport Name,0
Airport Country Code,0
Country Name,0
Airport Continent,0


## Renaming Columns

In [3]:
df = df.rename(columns={'Arrival Airport': 'Airport Code'})
display(df.head())

,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Continents,Departure Date,Airport Code,Pilot Name,Flight Status
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,6/28/2022,CXF,Fransisco Hazeldine,On Time
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,12/26/2022,YCO,Marla Parsonage,On Time
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,Europe,1/18/2022,GNB,Rhonda Amber,On Time
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,9/16/2022,YND,Kacie Commucci,Delayed
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2/25/2022,SEE,Ebonee Tree,On Time


## Processing Continent Columns

In [4]:
print("Unique values and counts for 'Airport Continent':")
display(df['Airport Continent'].value_counts())

print("\nUnique values and counts for 'Continents':")
display(df['Continents'].value_counts())

Unique values and counts for 'Airport Continent':


,count
Airport Continent,
NAM,32033
AS,18637
OC,13866
EU,12335
AF,11030
SAM,10718



Unique values and counts for 'Continents':


,count
Continents,
North America,32033
Asia,18637
Oceania,13866
Europe,12335
Africa,11030
South America,10718


Since Airport Continent and Continents are redundant, we consider deleting Continents column

In [5]:
df.drop(columns=['Continents'], inplace=True)

## Processing Date Column

In [6]:
print(f"Data type of 'Departure Date': {df['Departure Date'].dtype}")
print("First 10 unique values of 'Departure Date':")
display(df['Departure Date'].head(10))

Data type of 'Departure Date': object
First 10 unique values of 'Departure Date':


,Departure Date
0,6/28/2022
1,12/26/2022
2,1/18/2022
3,9/16/2022
4,2/25/2022
5,06-10-2022
6,10/30/2022
7,04-07-2022
8,8/20/2022
9,04-06-2022


The date column format is mixed. We convert all date values to dd/mm/yyyy

In [7]:
df['Departure Date'] = df['Departure Date'].str.replace('-', '/', regex=False)

# Convert to datetime so Python understands it as a date
df['Departure Date'] = pd.to_datetime(df['Departure Date'], dayfirst=False)

# Format it back to a string in dd/mm/yyyy
df['Departure Date'] = df['Departure Date'].dt.strftime('%d/%m/%Y')

In [8]:
df['Departure Date'].head(10)

,Departure Date
0,28/06/2022
1,26/12/2022
2,18/01/2022
3,16/09/2022
4,25/02/2022
5,10/06/2022
6,30/10/2022
7,07/04/2022
8,20/08/2022
9,06/04/2022


## Process Airport Code Column

In [9]:
import re

#in this part, we check which of the IATA code is not valid. Valid IATA code consists of 3 uppercased letters

#Valid IATA code = exactly 3 uppercase letters only
valid_iata = df['Airport Code'].str.match(r'^[A-Z]{3}$')

print(f"Valid iata codes:   {valid_iata.sum()}")
print(f"Invalid iata codes: {(~valid_iata).sum()}")
print(f"Percentage invalid: {((~valid_iata).sum() / len(df)) * 100:.2f}%")

print(f"\nInvalid codes found:")
print(df[~valid_iata]['Airport Code'].value_counts())

Valid iata codes:   97693
Invalid iata codes: 926
Percentage invalid: 0.94%

Invalid codes found:
Airport Code
0      873
SB2     16
TR7     11
O62     10
-        8
BR-      8
Name: count, dtype: int64


In [10]:
#We delete the rows of the airport with invalid IATA codes

#Drop invalid iata codes
valid_iata = df['Airport Code'].str.match(r'^[A-Z]{3}$')
df = df[valid_iata]

print(f"Rows after dropping invalid codes: {len(df)}")

Rows after dropping invalid codes: 97693


In [11]:
print("=== NULL VALUES ===")
print(df.isnull().sum())
print("\n=== NULL % ===")
print((df.isnull().mean() * 100).round(2))
print("\n=== DUPLICATE ROWS ===")
print(f"Total duplicates: {df.duplicated().sum()}")

=== NULL VALUES ===
Passenger ID            0
First Name              0
Last Name               0
Gender                  0
Age                     0
Nationality             0
Airport Name            0
Airport Country Code    0
Country Name            0
Airport Continent       0
Departure Date          0
Airport Code            0
Pilot Name              0
Flight Status           0
dtype: int64

=== NULL % ===
Passenger ID            0.0
First Name              0.0
Last Name               0.0
Gender                  0.0
Age                     0.0
Nationality             0.0
Airport Name            0.0
Airport Country Code    0.0
Country Name            0.0
Airport Continent       0.0
Departure Date          0.0
Airport Code            0.0
Pilot Name              0.0
Flight Status           0.0
dtype: float64

=== DUPLICATE ROWS ===
Total duplicates: 0


## Adding 'passenger_count' and 'is_international' column

In [12]:
#A passenger_count column was added to the dataset with a default value of 1 for each row, representing a single passenger per record.

df['passenger_count'] = 1

In [13]:
#Add a column to check if the passenger is an international flyer. If the passenger's nationality
#is not equal to the country name, fill 1 for international, else 0

import numpy as np

df['is_international'] = np.where(df['Nationality'] != df['Country Name'], 1, 0)

## Adding 'age_group' column

In [34]:
def get_age_group(age):
    if age < 18:   return 'Child (Under 18)'
    if age < 25:   return 'Young Adult (18-24)'
    if age < 45:   return 'Adult (25-44)'
    if age < 65:   return 'Middle Aged (45-64)'
    return 'Senior (65+)'

df['age_group'] = df['Age'].apply(get_age_group)

In [35]:
df.head(5)

,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Departure Date,Airport Code,Pilot Name,Flight Status,passenger_count,is_international,age_group
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,28/06/2022,CXF,Fransisco Hazeldine,On Time,1,1,Middle Aged (45-64)
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,26/12/2022,YCO,Marla Parsonage,On Time,1,1,Middle Aged (45-64)
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,18/01/2022,GNB,Rhonda Amber,On Time,1,1,Senior (65+)
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,16/09/2022,YND,Kacie Commucci,Delayed,1,1,Senior (65+)
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,25/02/2022,SEE,Ebonee Tree,On Time,1,1,Young Adult (18-24)


# Processing Countries dataset

In [17]:
countries_df = pd.read_csv('/content/countries.csv', keep_default_na=False)
display(countries_df.head())

,id,code,name,continent,wikipedia_link,keywords
0,302672,AD,Andorra,EU,https://en.wikipedia.org/wiki/Andorra,Andorran airports
1,302618,AE,United Arab Emirates,AS,https://en.wikipedia.org/wiki/United_Arab_Emir...,"UAE,مطارات في الإمارات العربية المتحدة"
2,302619,AF,Afghanistan,AS,https://en.wikipedia.org/wiki/Afghanistan,
3,302722,AG,Antigua and Barbuda,NA,https://en.wikipedia.org/wiki/Antigua_and_Barbuda,Antiguan airports
4,302723,AI,Anguilla,NA,https://en.wikipedia.org/wiki/Anguilla,


In [19]:
display(countries_df.isnull().sum())

# Count of duplicate rows
countries_df.duplicated().sum()

,0
id,0
code,0
name,0
continent,0
wikipedia_link,0
keywords,0


np.int64(0)

## Dropping Unused Columns

In [20]:
countries_df = countries_df.drop(columns=['keywords', 'wikipedia_link'], errors='ignore')
display(countries_df.head())

,id,code,name,continent
0,302672,AD,Andorra,EU
1,302618,AE,United Arab Emirates,AS
2,302619,AF,Afghanistan,AS
3,302722,AG,Antigua and Barbuda,NA
4,302723,AI,Anguilla,NA


# Processing Airport dataset

In [21]:
airports_df = pd.read_csv('/content/airports.csv', keep_default_na=True)
display(airports_df.head())

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [23]:
display(airports_df.isnull().sum())

display(airports_df.duplicated().sum())

,0
id,0
ident,0
type,0
name,0
latitude_deg,0
longitude_deg,0
elevation_ft,14664
continent,39453
iso_country,292
iso_region,0


np.int64(0)

## Dropping Unused Rows

In [24]:
columns_to_drop = ['icao_code', 'gps_code', 'local_code', 'home_link', 'wikipedia_link', 'keywords']
airports_df = airports_df.drop(columns=columns_to_drop, errors='ignore')
display(airports_df.head())

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,iata_code
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN


## Filling 'municipality' null values with 'Unknown'

In [25]:
airports_df['municipality'] = airports_df['municipality'].fillna('Unknown')

## Filling the null country values

In [26]:
#Strip the first 2 letters from iso_region, since the format is country-region

#fill the iso_country
airports_df['iso_country'] = airports_df['iso_country'].fillna(
    airports_df['iso_region'].str.split('-').str[0]
)


## Filling the null continent values

In [27]:
#We first build a lookup table from countries.csv, filled with the country code and continent code

#fill the continent
#Build a mapping of country code -> continent from countries.csv
code_to_continent = countries_df.set_index('code')['continent'].to_dict()

#Then we fill the missing values by doing a lookup from that dataset
#Fill missing continent values by looking up iso_country
airports_df['continent'] = airports_df['continent'].fillna(
    airports_df['iso_country'].map(code_to_continent)
)

## Filling the null elevation values

In [28]:
#Calculate mean elevation per continent
continent_mean_elevation = airports_df.groupby('continent')['elevation_ft'].mean()

#Fill missing elevation_ft with the mean of its continent
airports_df['elevation_ft'] = airports_df.groupby('continent')['elevation_ft'].transform(
    lambda x: x.fillna(x.mean())
)

## Processing the IATA code column

In [29]:
#Check which airport names from df exist in airports_df
df_names = set(df['Airport Name'].str.lower().str.strip())
airport_names = set(airports_df['name'].str.lower().str.strip())

matched_names = df_names & airport_names  # intersection
unmatched_names = df_names - airport_names  # in df but not in airports_df

print(f"Total unique airport names in airline dataset: {len(df_names)}")
print(f"Names that exist in airports_df:               {len(matched_names)}")
print(f"Names that don't exist in airports_df:         {len(unmatched_names)}")

Total unique airport names in airline dataset: 8979
Names that exist in airports_df:               7428
Names that don't exist in airports_df:         1551


In [30]:
#Build lookup from airline dataset: name -> Airport Code
name_to_iata = (
    df.assign(norm_name=df['Airport Name'].str.lower().str.strip())
    [['norm_name', 'Airport Code']]
    .drop_duplicates(subset='norm_name')
    .set_index('norm_name')['Airport Code']
    .to_dict()
)

#Fill null iata_code in airports_df where name matches
airports_df['norm_name'] = airports_df['name'].str.lower().str.strip()
airports_df['iata_code'] = airports_df.apply(
    lambda row: name_to_iata.get(row['norm_name'], row['iata_code'])
    if pd.isnull(row['iata_code']) else row['iata_code'],
    axis=1
)

#Check results
null_after = airports_df['iata_code'].isna().sum()
print(f"Null iata_codes before: 75477")
print(f"Null iata_codes after:  {null_after}")
print(f"Successfully filled:    {75477 - null_after}")

# Clean up
airports_df = airports_df.drop(columns=['norm_name'])

Null iata_codes before: 75477
Null iata_codes after:  74444
Successfully filled:    1033


In [31]:
#Drop remaining null iata_code rows
airports_df = airports_df[airports_df['iata_code'].notna()]
print(f"airports_df final rows: {len(airports_df)}")

airports_df final rows: 10092


In [32]:
display(airports_df.isnull().sum())

,0
id,0
ident,0
type,0
name,0
latitude_deg,0
longitude_deg,0
elevation_ft,0
continent,0
iso_country,0
iso_region,0


# Exporting the dataset to csv

In [36]:
df.to_csv('Airline_clean.csv', index=False)
countries_df.to_csv('Countries_clean.csv', index=False)
airports_df.to_csv('Airport_clean.csv', index=False)

print("All 3 dataframes exported successfully!")

All 3 dataframes exported successfully!
